## 1. Setup and imports

Import the required libraries for data loading, image processing, PyTorch training, and progress tracking.

In [1]:
import os
import random
import numpy as np
import pandas as pd

from PIL import Image

import torch
import torch.nn as nn
import torch.optim as optim

from torch.utils.data import Dataset
from torch.utils.data import DataLoader
from torch.utils.data import random_split

from torchvision import transforms

from sklearn.metrics import accuracy_score

from tqdm import tqdm

In [2]:
torch.backends.cudnn.benchmark = True

## 2. Configuration

Define dataset paths, model output directories, image size, batch size, number of epochs, learning rate, and random seed.

In [3]:
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "data").exists():
    PROJECT_ROOT = Path(r"f:\Samir\Projects\SnakeSense").resolve()

TRAIN_DIR = str(PROJECT_ROOT / "data" / "Processed_Images" / "Processed_Train")
VALID_DIR = str(PROJECT_ROOT / "data" / "Processed_Images" / "Processed_valid")
TEST_DIR = str(PROJECT_ROOT / "data" / "Processed_Images" / "Processed_test")

TRAIN_CSV = str(PROJECT_ROOT / "data" / "Processed_Images" / "Processed_Train" / "Processed_train_annotations.csv")
VALID_CSV = str(PROJECT_ROOT / "data" / "Processed_Images" / "Processed_valid" / "Processed_valid_annotations.csv")
TEST_CSV = str(PROJECT_ROOT / "data" / "Processed_Images" / "Processed_test" / "Processed_test_annotations.csv")

MODEL_DIR = str(PROJECT_ROOT / "models")
os.makedirs(MODEL_DIR, exist_ok=True)

In [4]:
IMAGE_SIZE = 224
BATCH_SIZE = 64
EPOCHS = 20
LEARNING_RATE = 5e-5
TRAIN_RATIO = 0.8
RANDOM_SEED = 42
WEIGHT_DECAY = 5e-5
PATIENCE = 5
NUM_WORKERS = 0

In [5]:
DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print(DEVICE)

cuda


## 3. Data augmentation and preprocessing

Create separate transforms for training and validation/test data.
Training uses stronger augmentation such as flipping, rotation, and color jitter.

In [6]:
from torchvision.models import efficientnet_b0, EfficientNet_B0_Weights

weights = EfficientNet_B0_Weights.IMAGENET1K_V1

IMAGENET_MEAN = weights.meta.get("mean", [0.485, 0.456, 0.406])
IMAGENET_STD = weights.meta.get("std", [0.229, 0.224, 0.225])

In [7]:
# 1. Strong Train Transforms (Breaks memorization & fixes overfitting)
train_transform = transforms.Compose([
    # Scale & Crop: forces the network to detect snakes from partial bodies/heads
    transforms.RandomResizedCrop((224, 224), scale=(0.7, 1.0)),
    # Orientation variations
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.2),
    transforms.RandomRotation(degrees=30),
    # Lighting & background ground-cover variations
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.ToTensor(),
    # ImageNet Standard Normalization
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

# 2. Validation Transform (Deterministic testing)
val_transform = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.CenterCrop((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

## 4. Dataset and data loaders

Define the custom `SnakeDataset` class and create train, validation, and test data loaders.

In [42]:
class SnakeDataset(Dataset):

    def __init__(self, csv_path, image_folder, transform=None):
        """
        Args:
            csv_path (str): Path to processed_annotations.csv
            image_folder (str): Folder containing cropped images
            transform (callable): torchvision transforms
        """

        self.annotations = pd.read_csv(csv_path)
        self.image_folder = image_folder
        self.transform = transform

    def __len__(self):
        return len(self.annotations)

    def __getitem__(self, index):

        row = self.annotations.iloc[index]

        image_path = os.path.join(
            self.image_folder,
            row["filename"]
        )

        if not os.path.exists(image_path):
            raise FileNotFoundError(f"Image not found:\n{image_path}")

        with Image.open(image_path) as image:
            image = image.convert("RGB")

            if self.transform is not None:
                image = self.transform(image)

        label = int(row["Label"])
        
        return image, label

    @property
    def classes(self):
        for col in ["Class", "class", "class_name", "Class_Name", "ClassName"]:
            if col in self.annotations.columns:
                return sorted(self.annotations[col].astype(str).dropna().unique().tolist())

        if "Label" in self.annotations.columns:
            labels = sorted(self.annotations["Label"].astype(int).dropna().unique().tolist())
            return [f"Class_{label}" for label in labels]

        return [f"Class_{i}" for i in range(self.num_classes)]

    @property
    def num_classes(self):
        return self.annotations["Label"].nunique()

In [43]:
train_dataset = SnakeDataset(
    csv_path=TRAIN_CSV,
    image_folder=TRAIN_DIR,
    transform=train_transform
)

valid_dataset = SnakeDataset(
    csv_path=VALID_CSV,
    image_folder=VALID_DIR,
    transform=val_transform
)

test_dataset = SnakeDataset(
    csv_path=TEST_CSV,
    image_folder=TEST_DIR,
    transform=val_transform
)

In [44]:
print(f"Train Images      : {len(train_dataset)}")
print(f"Validation Images : {len(valid_dataset)}")
print(f"Test Images       : {len(test_dataset)}")

print(f"Classes           : {train_dataset.num_classes}")

Train Images      : 6108
Validation Images : 572
Test Images       : 290
Classes           : 15


In [45]:
train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0,  # Set to 0 to prevent OS multiprocessing locks
    pin_memory=True
)

valid_loader = DataLoader(
    valid_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=(DEVICE.type == "cuda"),
    persistent_workers=False
)

In [46]:
print(f"Train samples: {len(train_dataset)} | Valid samples: {len(valid_dataset)}")

Train samples: 6108 | Valid samples: 572


In [47]:
images, labels = next(iter(train_loader))
print(images.shape, labels.shape)

torch.Size([64, 3, 224, 224]) torch.Size([64])


## 5. Data sanity checks

Verify the number of samples, class count, and sample batch shape before training.
These checks help confirm that the data pipeline is correct.

In [48]:
# images, labels = next(iter(train_loader))

# print(images.shape)

In [49]:
print(len(train_dataset))
print(len(valid_dataset))

6108
572


In [50]:
import platform
print(platform.processor())

Intel64 Family 6 Model 141 Stepping 1, GenuineIntel


## 6. Model definition



In [51]:
from torchvision.models import efficientnet_b0, EfficientNet_B0_Weights

weights = EfficientNet_B0_Weights.IMAGENET1K_V1

model = efficientnet_b0(weights=weights)

num_features = model.classifier[-1].in_features

model.classifier[-1] = nn.Linear(
    num_features,
    15
)

model.to(DEVICE)

print("=" * 50)
print("Model Loaded Successfully")
print("=" * 50)
print(f"Architecture : EfficientNet B0")
print(f"Classes      : {15}")
print(f"Device       : {DEVICE}")

Model Loaded Successfully
Architecture : EfficientNet B0
Classes      : 15
Device       : cuda


## 7. Loss, optimizer, and scheduler

Define the loss function, optimizer, learning-rate scheduler, and mixed-precision scaler for training.

In [52]:
criterion = nn.CrossEntropyLoss(
    label_smoothing=0.05
)

In [53]:
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY
)

In [54]:
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=EPOCHS
)

In [55]:
scaler = torch.cuda.amp.GradScaler(enabled=(DEVICE.type == "cuda"))

C:\Users\kisha\AppData\Local\Temp\ipykernel_14388\544004983.py:1: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=(DEVICE.type == "cuda"))


In [56]:
print("=" * 50)
print("Training Configuration")
print("=" * 50)

print(f"Model         : EfficientNet B0")
print(f"Classes       : {15}")
print(f"Image Size    : {IMAGE_SIZE}")
print(f"Batch Size    : {BATCH_SIZE}")
print(f"Epochs        : {EPOCHS}")
print(f"Learning Rate : {LEARNING_RATE}")
print(f"Device        : {DEVICE}")
print(f"Optimizer     : {optimizer.__class__.__name__}")
print(f"Scheduler     : {scheduler.__class__.__name__}")
print(f"Loss          : {criterion.__class__.__name__}")
print("=" * 50)

Training Configuration
Model         : EfficientNet B0
Classes       : 15
Image Size    : 224
Batch Size    : 64
Epochs        : 20
Learning Rate : 5e-05
Device        : cuda
Optimizer     : AdamW
Scheduler     : CosineAnnealingLR
Loss          : CrossEntropyLoss


## 8. Training and validation helpers

These functions run one epoch of training and one epoch of validation while tracking loss and accuracy.

In [57]:
CHECKPOINT_DIR = str(PROJECT_ROOT / "checkpoints")
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

LAST_CHECKPOINT = os.path.join(CHECKPOINT_DIR, "last_checkpoint.pth")
BEST_MODEL = os.path.join(CHECKPOINT_DIR, "best_model.pth")

In [58]:
def train_one_epoch(model, loader, criterion, optimizer, scaler, device):
    model.train()

    running_loss = 0.0
    correct = 0
    total = 0

    progress_bar = tqdm(
        loader,
        total=len(loader),
        desc="Training",
        dynamic_ncols=True,
        leave=True,
        ncols=120
    )

    for batch_idx, (images, labels) in enumerate(progress_bar):
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        with torch.autocast(device_type=device.type, enabled=(device.type == "cuda")):
            outputs = model(images)
            loss = criterion(outputs, labels)

        if device.type == "cuda":
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            scaler.step(optimizer)
            scaler.update()
        else:
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()

        running_loss += loss.detach().item()

        predictions = outputs.argmax(dim=1)
        correct += (predictions == labels).sum().item()
        total += labels.size(0)

        avg_loss = running_loss / (batch_idx + 1)
        avg_acc = 100.0 * correct / total

        progress_bar.set_postfix(
            Loss=f"{avg_loss:.4f}",
            Acc=f"{avg_acc:.2f}%"
        )

    epoch_loss = running_loss / len(loader)
    epoch_acc = 100.0 * correct / total

    return epoch_loss, epoch_acc

In [59]:
@torch.no_grad()
def validate(model, loader, criterion, device):
    model.eval()

    running_loss = 0.0
    correct = 0
    total = 0

    progress_bar = tqdm(
        loader,
        total=len(loader),
        desc="Validation",
        dynamic_ncols=True,
        leave=True,
        ncols=120
    )

    for batch_idx, (images, labels) in enumerate(progress_bar):
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        with torch.autocast(device_type=device.type, enabled=(device.type == "cuda")):
            outputs = model(images)
            loss = criterion(outputs, labels)

        running_loss += loss.detach().item()

        predictions = outputs.argmax(dim=1)
        correct += (predictions == labels).sum().item()
        total += labels.size(0)

        avg_loss = running_loss / (batch_idx + 1)
        avg_acc = 100.0 * correct / total

        progress_bar.set_postfix(
            Loss=f"{avg_loss:.4f}",
            Acc=f"{avg_acc:.2f}%"
        )

    epoch_loss = running_loss / len(loader)
    epoch_acc = 100.0 * correct / total

    return epoch_loss, epoch_acc

## 9. Checkpointing and resume

Save the latest checkpoint and the best model during training.
If a previous checkpoint exists, resume training from the saved state.

In [60]:
# ==========================
# Resume Training (Optional)
# ==========================

start_epoch = 0
best_accuracy = 0.0

history = {
    "train_loss": [],
    "train_acc": [],
    "valid_loss": [],
    "valid_acc": []
}

if os.path.exists(LAST_CHECKPOINT):

    print("=" * 60)
    print("Resuming Training")
    print("=" * 60)

    checkpoint = torch.load(
        LAST_CHECKPOINT,
        map_location=DEVICE
    )

    model.load_state_dict(
        checkpoint["model_state_dict"]
    )

    optimizer.load_state_dict(
        checkpoint["optimizer_state_dict"]
    )

    scheduler.load_state_dict(
        checkpoint["scheduler_state_dict"]
    )

    scaler.load_state_dict(
        checkpoint["scaler_state_dict"]
    )

    start_epoch = checkpoint["epoch"] + 1
    best_accuracy = checkpoint["best_accuracy"]

    print(f"Resuming from Epoch : {start_epoch}")
    print(f"Best Accuracy       : {best_accuracy:.2f}%")
    print("=" * 60)

else:

    print("=" * 60)
    print("No checkpoint found.")
    print("Training will start from scratch.")
    print("=" * 60)

Resuming Training
Resuming from Epoch : 20
Best Accuracy       : 69.06%


In [61]:
train_df = pd.read_csv(TRAIN_CSV)

train_df = train_df.dropna().reset_index(drop=True)

train_df["Label"] = train_df["Label"].astype(int)

train_df.to_csv(TRAIN_CSV, index=False)

print("CSV repaired.")
print("Rows:", len(train_df))

CSV repaired.
Rows: 6108


In [62]:
train_df = pd.read_csv(TRAIN_CSV)

print(train_df.isna().sum())

filename    0
width       0
height      0
class       0
xmin        0
ymin        0
xmax        0
ymax        0
Label       0
dtype: int64


In [63]:
missing = []

for file in train_df["filename"]:

    if not os.path.exists(os.path.join(TRAIN_DIR, file)):
        missing.append(file)

print("Missing:", len(missing))

Missing: 0


## 10. Training loop

Train the model for all epochs, log training and validation metrics, and save the best model.

In [64]:
import time 

for epoch in range(start_epoch, EPOCHS):

    print("\n" + "=" * 60)
    print(f"Epoch {epoch + 1}/{EPOCHS}")
    print("=" * 60)

    # ==========================
    # Training
    # ==========================

    train_loss, train_acc = train_one_epoch(
        model=model,
        loader=train_loader,
        criterion=criterion,
        optimizer=optimizer,
        scaler=scaler,
        device=DEVICE
    )

    # ==========================
    # Validation
    # ==========================

    valid_loss, valid_acc = validate(
        model=model,
        loader=valid_loader,
        criterion=criterion,
        device=DEVICE
    )

    # ==========================
    # Save History
    # ==========================

    history["train_loss"].append(train_loss)
    history["train_acc"].append(train_acc)
    history["valid_loss"].append(valid_loss)
    history["valid_acc"].append(valid_acc)

    # ==========================
    # Update Learning Rate
    # ==========================

    scheduler.step()

    # ==========================
    # Epoch Summary
    # ==========================

    print(f"\nTrain Loss : {train_loss:.4f}")
    print(f"Train Acc  : {train_acc:.2f}%")
    print(f"Valid Loss : {valid_loss:.4f}")
    print(f"Valid Acc  : {valid_acc:.2f}%")

    # ==========================
    # Save Last Checkpoint
    # ==========================

    torch.save(
        {
            "epoch": epoch,
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "scheduler_state_dict": scheduler.state_dict(),
            "scaler_state_dict": scaler.state_dict(),
            "best_accuracy": best_accuracy,
        },
        LAST_CHECKPOINT
    )

    # ==========================
    # Save Best Model
    # ==========================

    if valid_acc > best_accuracy:

        best_accuracy = valid_acc

        torch.save(
            {
                "epoch": epoch,
                "model_state_dict": model.state_dict(),
                "optimizer_state_dict": optimizer.state_dict(),
                "scheduler_state_dict": scheduler.state_dict(),
                "scaler_state_dict": scaler.state_dict(),
                "best_accuracy": best_accuracy,
            },
            BEST_MODEL
        )

        print(f"✅ Best model saved ({best_accuracy:.2f}%)")

    else:

        print(f"No improvement (Best: {best_accuracy:.2f}%)")

## Load Best Model Checkpoint


In [65]:
checkpoint = torch.load(BEST_MODEL, map_location=DEVICE)
model.load_state_dict(checkpoint["model_state_dict"])
model.to(DEVICE)
model.eval()

print("==================================================")
print("✅ Best Model Loaded Successfully!")
print(f"Best Validation Accuracy : {checkpoint['best_accuracy']:.2f}%")
print(f"Saved Epoch              : {checkpoint['epoch'] + 1}")
print("==================================================")

✅ Best Model Loaded Successfully!
Best Validation Accuracy : 69.06%
Saved Epoch              : 19


## Evaluate on the Unseen Test Set

In [66]:
@torch.no_grad()
def evaluate_test_set(model, loader, criterion, device):
    model.eval()
    running_loss = 0.0
    correct_top1 = 0
    correct_top3 = 0
    total = 0

    for images, labels in tqdm(loader, desc="Testing Set Evaluation", ncols=100):
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        with torch.autocast(device_type=device.type, enabled=(device.type == "cuda")):
            outputs = model(images)
            loss = criterion(outputs, labels)

        running_loss += loss.detach().item()

        # Top-1 Accuracy
        preds_top1 = outputs.argmax(dim=1)
        correct_top1 += (preds_top1 == labels).sum().item()

        # Top-3 Accuracy
        _, preds_top3 = outputs.topk(k=3, dim=1)
        for i in range(labels.size(0)):
            if labels[i] in preds_top3[i]:
                correct_top3 += 1

        total += labels.size(0)

    test_loss = running_loss / len(loader)
    test_acc_top1 = 100.0 * correct_top1 / total
    test_acc_top3 = 100.0 * correct_top3 / total

    return test_loss, test_acc_top1, test_acc_top3

test_loss, test_acc1, test_acc3 = evaluate_test_set(model, test_loader, criterion, DEVICE)

print("\n" + "=" * 50)
print("📊 TEST SET RESULTS")
print("=" * 50)
print(f"Test Loss      : {test_loss:.4f}")
print(f"Top-1 Accuracy : {test_acc1:.2f}%")
print(f"Top-3 Accuracy : {test_acc3:.2f}%")
print("=" * 50)

Testing Set Evaluation: 100%|█████████████████████████████████████████| 5/5 [00:02<00:00,  2.23it/s]


📊 TEST SET RESULTS
Test Loss      : 1.2008
Top-1 Accuracy : 70.00%
Top-3 Accuracy : 87.24%


In [67]:
import json

# Extract class list from train_dataset
class_names = train_dataset.classes
class_mapping = {int(idx): str(cls_name) for idx, cls_name in enumerate(class_names)}

mapping_path = Path(MODEL_DIR) / "class_mapping.json"

with open(mapping_path, "w") as f:
    json.dump(class_mapping, f, indent=4)

print("==================================================")
print(f"✅ Class mapping saved to: {mapping_path}")
print("==================================================")
print(json.dumps(class_mapping, indent=2))

✅ Class mapping saved to: F:\Samir\Projects\SnakeSense\models\class_mapping.json
{
  "0": "ahaetulla prasina",
  "1": "amphiesma stolatum",
  "2": "bungarus caeruleus",
  "3": "bungarus fasciatus",
  "4": "chrysopelea ornata",
  "5": "daboia russelii",
  "6": "dendrelaphis pictus",
  "7": "naja naja",
  "8": "ophiophagus hannah",
  "9": "psammodynastes pulverulentus",
  "10": "ptyas korros",
  "11": "ptyas mucosa",
  "12": "trimeresurus albolabris",
  "13": "trimeresurus purpureomaculatus",
  "14": "xenochrophis piscator"
}


In [68]:
import json
from PIL import Image

def predict_single_image(image_path, model, transform, class_mapping, device, top_k=3):
    model.eval()
    
    # 1. Open and convert image to RGB
    image = Image.open(image_path).convert("RGB")
    
    # 2. Apply validation transform and add batch dimension
    tensor_img = transform(image).unsqueeze(0).to(device)
    
    # 3. Forward pass
    with torch.no_grad():
        outputs = model(tensor_img)
        probabilities = torch.softmax(outputs, dim=1)
        top_probs, top_indices = torch.topk(probabilities, k=top_k)
        
    # 4. Format top-3 predictions
    results = []
    for rank, (prob, idx) in enumerate(zip(top_probs[0], top_indices[0]), start=1):
        class_name = class_mapping.get(idx.item(), "Unknown")
        results.append({
            "rank": rank,
            "species": class_name,
            "confidence": f"{prob.item() * 100:.2f}%"
        })
        
    return results

# Pick the first sample image from your TEST_DIR
sample_image_file = os.listdir(TEST_DIR)[0]
sample_image_path = os.path.join(TEST_DIR, sample_image_file)

# Convert integer keys mapping for lookup safety
mapping_dict = {int(k): v for k, v in class_mapping.items()}

# Run test prediction
test_prediction = predict_single_image(
    image_path=sample_image_path,
    model=model,
    transform=val_transform,
    class_mapping=mapping_dict,
    device=DEVICE
)

print("==================================================")
print(f"📷 Testing Image : {sample_image_file}")
print("==================================================")
print(json.dumps(test_prediction, indent=2))
print("==================================================")

📷 Testing Image : 100346824_jpeg.rf.37cdc3e14cb9f0f1a770d8bb470abb42.jpg
[
  {
    "rank": 1,
    "species": "ptyas korros",
    "confidence": "37.00%"
  },
  {
    "rank": 2,
    "species": "naja naja",
    "confidence": "34.15%"
  },
  {
    "rank": 3,
    "species": "ptyas mucosa",
    "confidence": "24.90%"
  }
]
